[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reinhart-group/generative-copolymer-workshop/blob/main/day1/01_md_simulation.ipynb)

# Day 1 — Morning: MD Simulation Basics

**Objectives:**
- Understand the basics of molecular dynamics: integrators, thermostats, and potentials
- Learn why we coarse-grain: the Kremer-Grest model
- Initialize a HOOMD-blue simulation environment
- Define sequence architectures (diblock, alternating) and run a short equilibration

In [ ]:
#@title  ⚙️  Step 1 of 2 — install conda (causes a restart; that's normal). { display-mode: "form" }
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
#@title  ⚙️  Step 2 of 2 — install the simulation + drawing packages (~2 min). { display-mode: "form" }
%%capture
!conda install -y python=3.12 scipy gsd "hoomd=*=cpu*" freud fresnel pillow
!pip install -q plotly

In [ ]:
#@title  ⚙️  cdse_lab — shared plotting/IO helpers for Day 1. You can ignore this cell. { display-mode: "form" }
#
# This module intentionally contains NO simulation code. HOOMD snapshot setup,
# integrators, thermostats, and forces are written out directly in each notebook
# (01_md_simulation, 02_virtual_lab) -- seeing that code is the lesson. This
# module only hides generic, non-pedagogical plumbing:
#
#     Trajectory                              -- a lightweight "movie": frames + times
#     grab_frame(snapshot)                    -- capture a HOOMD/GSD snapshot into a frame dict
#     unwrap(frame)                           -- undo periodic wrapping (image flags -> continuous coords)
#     save_gsd(traj, filename) / load_gsd(f)  -- write / read a real .gsd file
#     show3d(frame_or_traj) / animate3d(traj) -- interactive 3D picture / movie (Plotly)
#     show_row(*figs)                         -- lay several show3d/animate3d figures side by side
#     measure_size(traj)                      -- radius of gyration vs. time
#     render_pretty(frame_or_traj)            -- optional ray-traced still (needs fresnel)
#
# Forked from reu/cdse_lab.py (a separate, unrelated curriculum) -- this copy is
# scoped to the workshop and does not track that file.

import numpy as np
import gsd.hoomd
import plotly.graph_objects as go

# Colors: type "A" = red (slippery), type "B" = blue (sticky)
_A_COLOR = "crimson"
_B_COLOR = "royalblue"

__all__ = [
    "Trajectory", "grab_frame", "unwrap",
    "save_gsd", "load_gsd",
    "show3d", "show_row", "animate3d", "measure_size", "render_pretty",
]


def _typeids_to_seq(typeids):
    """Inverse of the 'A'/'B' -> 0/1 encoding, for one chain's worth of typeids."""
    return "".join("A" if int(t) == 0 else "B" for t in typeids)


class Trajectory:
    """A lightweight 'movie': a list of frames plus a little bookkeeping.

    Each frame is a dict with:
        position : (N,3) float  -- WRAPPED coordinates (inside the box), as in a real GSD
        image    : (N,3) int    -- how many box-lengths each particle has wandered
        typeid   : (N,)  int     -- 0 = A, 1 = B
        bonds    : (M,2) int     -- pairs of bonded particle indices
        box      : (3,)  float   -- box edge lengths (Lx, Ly, Lz)
    Use unwrap(frame) to get continuous (un-wrapped) coordinates.
    """

    def __init__(self, sequence, num_chains):
        self.sequence = sequence
        self.num_chains = num_chains
        self.chain_len = len(sequence)
        self.frames = []
        self.times = []

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, i):
        return self.frames[i]

    def __repr__(self):
        return (f"Trajectory(sequence={self.sequence!r}, num_chains={self.num_chains}, "
                f"frames={len(self.frames)})")


def grab_frame(snapshot):
    """Copy the bits we need out of a HOOMD/GSD snapshot (wrapped coords + image flags)."""
    return {
        "position": np.array(snapshot.particles.position, dtype=float),
        "image": np.array(snapshot.particles.image, dtype=int),
        "typeid": np.array(snapshot.particles.typeid, dtype=int),
        "bonds": (np.array(snapshot.bonds.group, dtype=int)
                  if snapshot.bonds.N else np.zeros((0, 2), int)),
        "box": np.array(snapshot.configuration.box[:3], dtype=float),
    }


def unwrap(frame):
    """Return continuous (un-wrapped) coordinates: position + image * box.

    HOOMD stores every particle *inside* the box (wrapped) plus an integer
    "image" counting how many box-lengths it has crossed. Undoing the wrap keeps
    bonded chains continuous instead of jumping across the periodic boundary.
    """
    return frame["position"] + frame["image"] * frame["box"]


# --------------------------------------------------------------------------- #
#  GSD files
# --------------------------------------------------------------------------- #
def save_gsd(traj, filename):
    """Write a Trajectory to a real HOOMD GSD file.

    Static topology (types, bonds) is stored once in frame 0; per-particle
    positions and image flags are stored every frame, exactly like HOOMD does.
    """
    with gsd.hoomd.open(name=filename, mode="w") as gsd_file:
        for i, fr in enumerate(traj.frames):
            f = gsd.hoomd.Frame()
            f.configuration.step = int(traj.times[i])
            L = fr["box"]
            f.configuration.box = [L[0], L[1], L[2], 0, 0, 0]
            f.particles.N = int(len(fr["position"]))
            f.particles.position = fr["position"]
            f.particles.image = fr["image"]
            f.particles.typeid = fr["typeid"]
            if i == 0:
                f.particles.types = ["A", "B"]
                nb = int(len(fr["bonds"]))
                f.bonds.N = nb
                f.bonds.types = ["bond"]
                if nb:
                    f.bonds.group = fr["bonds"]
                    f.bonds.typeid = np.zeros(nb, dtype=int)
            gsd_file.append(f)
    return filename


def load_gsd(filename):
    """Read a HOOMD GSD file back into a Trajectory.

    Reconstructs the sequence and chain count from the topology (works for the
    linear chains this course uses: num_chains = N - n_bonds, chain_len = N / num_chains).
    """
    with gsd.hoomd.open(name=filename, mode="r") as gsd_file:
        first = gsd_file[0]
        N = int(first.particles.N)
        bonds0 = (np.array(first.bonds.group, dtype=int)
                  if first.bonds.N else np.zeros((0, 2), int))
        n_bonds = len(bonds0)
        num_chains = max(1, N - n_bonds)
        chain_len = N // num_chains
        typeid0 = np.array(first.particles.typeid, dtype=int)
        sequence = _typeids_to_seq(typeid0[:chain_len])

        traj = Trajectory(sequence, num_chains)
        for frame in gsd_file:
            box = np.array(frame.configuration.box[:3], dtype=float)
            traj.frames.append({
                "position": np.array(frame.particles.position, dtype=float),
                "image": np.array(frame.particles.image, dtype=int),
                "typeid": np.array(frame.particles.typeid, dtype=int),
                "bonds": bonds0,
                "box": box,
            })
            traj.times.append(int(frame.configuration.step))
    return traj


# --------------------------------------------------------------------------- #
#  Visualization
# --------------------------------------------------------------------------- #
def _frame_traces(frame):
    pos = unwrap(frame)
    tid, bonds = frame["typeid"], frame["bonds"]

    bx, by, bz = [], [], []
    for a, b in bonds:
        bx += [pos[a, 0], pos[b, 0], None]
        by += [pos[a, 1], pos[b, 1], None]
        bz += [pos[a, 2], pos[b, 2], None]

    colors = [_A_COLOR if t == 0 else _B_COLOR for t in tid]
    bond_trace = go.Scatter3d(x=bx, y=by, z=bz, mode="lines",
                              line=dict(color="lightgray", width=4),
                              hoverinfo="skip", showlegend=False)
    bead_trace = go.Scatter3d(x=pos[:, 0], y=pos[:, 1], z=pos[:, 2], mode="markers",
                              marker=dict(size=5, color=colors),
                              hoverinfo="skip", showlegend=False)
    return [bond_trace, bead_trace]


def _axis_ranges(frames):
    allpos = np.concatenate([unwrap(f) for f in frames], axis=0)
    lo, hi = allpos.min(axis=0), allpos.max(axis=0)
    pad = 0.1 * (hi - lo + 1.0)
    return [(lo[i] - pad[i], hi[i] + pad[i]) for i in range(3)]


def _scene(ranges):
    xr, yr, zr = ranges
    return dict(xaxis=dict(range=xr, visible=False),
                yaxis=dict(range=yr, visible=False),
                zaxis=dict(range=zr, visible=False),
                aspectmode="data")


def show3d(obj, title=None, return_fig=False):
    """Draw ONE interactive 3D picture. Pass a frame OR a Trajectory (shows last frame).

    return_fig=True returns the Figure instead of showing it, so you can hand
    several to show_row(...) and see them side by side in a single output.
    """
    frame = obj.frames[-1] if isinstance(obj, Trajectory) else obj
    fig = go.Figure(data=_frame_traces(frame))
    fig.update_layout(scene=_scene(_axis_ranges([frame])),
                      margin=dict(l=0, r=0, t=30 if title else 0, b=0),
                      title=title, width=600, height=500)
    if return_fig:
        return fig
    fig.show()


def show_row(*figs, width=440, height=430, gap=8):
    """Show several Plotly figures SIDE BY SIDE in a single output (no vertical stacking).

    Pass figures made with show3d(..., return_fig=True) / animate3d(..., return_fig=True),
    or any go.Figure. Colab caps the output height but allows width, so this lays the
    figures in one horizontal flex row (scrolls sideways if they overflow).
    """
    from IPython.display import HTML, display

    blocks = []
    for k, fig in enumerate(figs):
        fig.update_layout(width=width, height=height)
        blocks.append(fig.to_html(full_html=False,
                                  include_plotlyjs=("cdn" if k == 0 else False)))
    cells = "".join(f'<div style="flex:0 0 auto;">{b}</div>' for b in blocks)
    display(HTML(
        f'<div style="display:flex;flex-wrap:nowrap;gap:{gap}px;'
        f'overflow-x:auto;align-items:flex-start;">{cells}</div>'))


def animate3d(traj, title=None, return_fig=False):
    """Draw a 3D MOVIE with a play button and a slider.

    return_fig=True returns the Figure instead of showing it (see show_row).
    """
    ranges = _axis_ranges(traj.frames)
    fig = go.Figure(
        data=_frame_traces(traj.frames[0]),
        frames=[go.Frame(data=_frame_traces(f), name=str(i))
                for i, f in enumerate(traj.frames)],
    )
    fig.update_layout(
        scene=_scene(ranges),
        margin=dict(l=0, r=0, t=30 if title else 0, b=0),
        title=title, width=600, height=520,
        updatemenus=[dict(type="buttons", showactive=False, x=0.05, y=0.05, xanchor="left",
            buttons=[
                dict(label="▶ Play", method="animate",
                     args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True)]),
                dict(label="⏸ Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
            ])],
        sliders=[dict(active=0, x=0.15, len=0.8, y=0,
            steps=[dict(method="animate", label=str(i),
                        args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
                   for i in range(len(traj.frames))])],
    )
    if return_fig:
        return fig
    fig.show()


def measure_size(traj):
    """Return (times, Rg): radius of gyration averaged over chains, per frame."""
    cl, nc = traj.chain_len, traj.num_chains
    times, sizes = [], []
    for frame, t in zip(traj.frames, traj.times):
        pos = unwrap(frame).reshape(nc, cl, 3)
        com = pos.mean(axis=1, keepdims=True)
        rg2 = ((pos - com) ** 2).sum(axis=2).mean(axis=1)
        sizes.append(np.sqrt(rg2).mean())
        times.append(t)
    return np.array(times), np.array(sizes)


def render_pretty(obj):
    """Optional: a fancy ray-traced still image (needs fresnel). Pass a frame or trajectory."""
    import fresnel, PIL.Image

    frame = obj.frames[-1] if isinstance(obj, Trajectory) else obj
    pos = unwrap(frame)
    tid, bonds = frame["typeid"], frame["bonds"]
    N = len(pos)

    scene = fresnel.Scene()
    colors = np.empty((N, 3))
    colors[tid == 0] = fresnel.color.linear([0.95, 0, 0])
    colors[tid == 1] = fresnel.color.linear([0, 0, 0.95])
    geo = fresnel.geometry.Sphere(scene, N=N, radius=0.3)
    geo.position[:] = pos
    geo.material = fresnel.material.Material(roughness=0.9)
    geo.outline_width = 0.05
    geo.material.primitive_color_mix = 1.0
    geo.color[:] = fresnel.color.linear(colors)

    if len(bonds):
        ends = np.stack([pos[bonds[:, 0]], pos[bonds[:, 1]]], axis=1)
        cyl = fresnel.geometry.Cylinder(scene, N=len(bonds))
        cyl.material = fresnel.material.Material(
            roughness=0.5, color=fresnel.color.linear([0.8, 0.8, 0.8]))
        cyl.points[:] = ends
        cyl.radius[:] = [0.12] * len(bonds)

    scene.background_color = (1, 1, 1)
    scene.background_alpha = 1
    scene.camera = fresnel.camera.Orthographic.fit(scene, view="isometric", margin=0.1)
    out = fresnel.preview(scene, w=600, h=600)
    return PIL.Image.fromarray(out[:, :, 0:3], mode="RGB")

In [ ]:
import hoomd
import gsd.hoomd
import numpy as np
import matplotlib.pyplot as plt

What is a Molecular Dynamics (MD) Simulation?

* **Molecular Dynamics (MD)** is a computational method to simulate the **time evolution of atoms and molecules** by solving their equations of motion. [\[en.wikipedia.org\]](https://en.wikipedia.org/wiki/Molecular_dynamics)
* Atoms are treated as classical particles interacting via a **force field** (potential energy function) or a **model**.
* Based on **Newton’s Second Law**:

$$
m_i \frac{d^2 \mathbf{r}_i}{dt^2} = \mathbf{F}_i
$$

* Forces are derived from the **potential energy**:

$$
\mathbf{F}_i = -\nabla_i U(\{\mathbf{r}_j\})
$$


MD = “numerical experiment” tracking atoms step-by-step in time.

## Classical Molecular Dynamics Workflow

Basic Algorithm:
1. Initialize positions & velocities
2. Compute forces
3. Integrate equations of motion
4. Update positions & velocities
5. Repeat

This iterative loop produces atomic trajectories.


# Ensembles in MD

What is an Ensemble?
* A statistical description of many possible states consistent with thermodynamic constraints.
* Determines **what variables are fixed vs fluctuating**.

## Common Ensembles

**Microcanonical (NVE)**
* Constant:
  * Number of particles (N)
  * Volume (V)
  * Energy (E)
* Properties:
  * “Natural” MD ensemble
  * No external coupling
* Used for:
  * Pure dynamics studies

**Canonical (NVT)**
* Constant:
  * N, V, Temperature (T)
* Requires **thermostat**
* Represents system in contact with heat bath

**Isothermal-Isobaric (NPT)**
* Constant:
  * N, Pressure (P), Temperature (T)
* Requires:
  * Thermostat + Barostat



### MD Simulation Pipeline

1. Define:
   * Model
   * Initial configuration

2. Choose:
   * Ensemble (NVE/NVT/NPT)
   * Integrator (Velocity Verlet)
   * Thermostat/barostat

3. Run simulation:
   * Iterate time steps
   * Simulate long enough to be equilibrated

4. Analyze:
   * Structural and Dynamic Properties

# Key Takeaways

* MD simulates atomic motion via **Newton’s laws**
* **Ensembles** define thermodynamic conditions
* **Velocity Verlet** integrates motion efficiently
* **Thermostats** control temperature
* **Periodic boundaries** mimic bulk systems

# References & Further Reading

* Allen, M. P. & Tildesley, D. J.  *Computer Simulation of Liquids*
* Frenkel, D. & Smit, B.  *Understanding Molecular Simulation*
*  Best Practices for Foundations in Molecular Simulations, Living J Comput Mol Sci. 2018 Nov 29;1(1):5957. doi: 10.33011/livecoms.1.1.5957
* [Molecular Dynamics – Wikipedia](https://en.wikipedia.org/wiki/Molecular_dynamics)
* [Lecture Notes](https://stattlab.github.io/atomistic-scale-simulations/introduction/index.html)

# Run Actual Simulation

## Inital Configuration Setup

In [ ]:
import numpy as np
import hoomd
import gsd, gsd.hoomd

frame = gsd.hoomd.Frame()
frame.particles.N = 10
frame.configuration.box = [15, 15, 15, 0, 0, 0]
frame.particles.types = ['A']
frame.bonds.types = ['bond']

frame.particles.typeid = np.array([0,0,0,0,0,0,0,0,0,0])

frame.particles.position=np.array([[0,0,-2],
                          [0,0,-1],
                          [0,0,0],
                          [0,0,1],
                          [0,0,2],
                          [0,0,3],
                          [0,0,4],
                          [0,0,5],
                          [0,0,6],
                          [0,0,7]])
frame.bonds.N = 9
frame.bonds.group = np.array([[0,1],
                     [1,2],
                     [2,3],
                     [3,4],
                     [4,5],
                     [5,6],
                     [6,7],
                     [7,8],
                     [8,9]])

show3d(grab_frame(frame))

## Initializing HOOMD-blue
TODO: explain


In [ ]:
# Setup Simulation
cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=1)
simulation.create_state_from_snapshot(frame)



# Time Integration: Velocity Verlet Algorithm
We need to solve Newton’s equations numerically (no analytic solution for many-body systems)
## Velocity Verlet Scheme
1. Position update:
$$
\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\Delta t + \frac{1}{2}\mathbf{a}(t)\Delta t^2
$$
2. Half-step velocity:
$$
\mathbf{v}(t+\tfrac{1}{2}\Delta t) = \mathbf{v}(t) + \frac{1}{2}\mathbf{a}(t)\Delta t
$$
3. Compute new forces → $$\mathbf{a}(t+\Delta t)$$
4. Final velocity update:
$$
\mathbf{v}(t+\Delta t) = \mathbf{v}(t+\tfrac{1}{2}\Delta t) + \frac{1}{2}\mathbf{a}(t+\Delta t)\Delta t
$$


Why Velocity Verlet?
* Time-reversible
* Good energy conservation
* Efficient (only one force evaluation per step)
* Velocity verlet is the standard integrator in MD packages.


In [ ]:
integrator = hoomd.md.Integrator(dt=0.005)

# Thermostats

* Control temperature → simulate **canonical ensemble (NVT)**
* Ensure correct energy distribution (Maxwell–Boltzmann)
* Couple system to a **heat bath**
* Modify velocities
* Thermostats enforce **temperature control**
* Tradeoff: **impact on dynamics**


Langevin Thermostat:
  * Friction term
  * Random noise
TODO: add more details. We'll use Langevin
* Models solvent/environment

In [ ]:
all_particles = hoomd.filter.All()
langevin = hoomd.md.methods.Langevin(filter=all_particles, kT=0.5)
simulation.operations.integrator = integrator
integrator.methods.append(langevin)

## Particle Interactions & Neighbor list

In [ ]:
# Neighbor List
neigh = hoomd.md.nlist.Tree(buffer=0.4)

### Coarse-Graining
![schematic](../sources/day1/images/coarse_graining.jpeg)
ACS Omega 2021, 6, 3, 1758-1772
length and timescales -> level of detail matters
what phenomena are captured on what level
what inputs are needed for each level
how to group atoms/ desired resolution
pair interactions, bonds, angles

### Top Down Coarse Graining
Top Down Coarse Graining vs bottom up
Objective: use macroscopic/mesoscale information to parameterize a simplistic model or build simplistic model to match theory
![schematic](../sources/day1/images/model_exp_theory.png)

### The Kremer-Grest Model
Pair interactions, bond interaction
Model description and parameters


## References and Further Reading
* [Workshop on Mesoscale Modeling](https://github.com/icomse/11th_workshop_mesoscale_modeling/tree/main)

In [ ]:
lj = hoomd.md.pair.LJ(nlist=neigh, default_r_cut=3.0)
lj.mode = "shift"
lj.params[("A","A")] = dict(epsilon=1, sigma=1)
lj.r_cut[('A', 'A')] = 2**(1/6.)
integrator.forces.append(lj)

harmonic = hoomd.md.bond.Harmonic()
harmonic.params['bond'] = dict(k=100, r0=1)
integrator.forces.append(harmonic)

## Add Trajectory Writing
TODO: explain, add thermo quantities?

## Run First Simulation

In [ ]:
traj = Trajectory(sequence="A" * 10, num_chains=1)
traj.frames.append(grab_frame(simulation.state.get_snapshot()))
traj.times.append(simulation.timestep)
n_frames = 24
chunk = max(1, 10000 // n_frames)
for _ in range(n_frames):
    simulation.run(chunk)
    traj.frames.append(grab_frame(simulation.state.get_snapshot()))
    traj.times.append(simulation.timestep)
animate3d(traj)

# Periodic Boundary Conditions (PBC)

Challenge:
* Simulations have **finite size**
* Real systems are bulk/infinite

Solution: Periodic Boundary Conditions:
* Simulation box is replicated infinitely
* When a particle leaves, it re-enters from opposite side
* Minimum Image Convention: Each particle interacts only with the **closest periodic image**

Why PBC?
* Eliminates surface effects
* Mimics bulk behavior
* Small system behaves like macroscopic system

Visualization:
  * “Tiling space with identical copies of simulation box”

Important:
* Requires **cutoffs** for interactions (e.g. Lennard-Jones, no long ranged interactions like Coulomb)


What MD produces:
* A **trajectory**:
  * Positions $$\mathbf{r}_i(t)$$
  * Velocities $$\mathbf{v}_i(t)$$
* Enables calculation of:
  * Thermodynamic properties (T, P, energy)
  * Structural properties (RDF, conformations)
  * Dynamical properties (diffusion, viscosity)

In [ ]:
# trajectory - open with VMD, Ovito
gsd_writer = hoomd.write.GSD(
    mode = 'wb',
    trigger=hoomd.trigger.Periodic(10),
    dynamic=['property','momentum'],
    filename='single_chain.gsd',
)
simulation.operations.writers.append(gsd_writer)

# themrodyamic quantities
thermodynamic_properties = hoomd.md.compute.ThermodynamicQuantities(
    filter=hoomd.filter.All()
)
simulation.operations.computes.append(thermodynamic_properties)
logger = hoomd.logging.Logger(categories=["scalar"])
logger.add( simulation, quantities=["timestep"])
logger.add(thermodynamic_properties,quantities=["pressure","kinetic_temperature"])
table = hoomd.write.Table(trigger=hoomd.trigger.Periodic(period=10), logger=logger)
simulation.operations.writers.append(table)

simulation.run(1000)
# #

## Many polymers at finite concentration

In [ ]:

del simulation

length_polymer = 10
N_polymer = 30
L = 22

N_particles = N_polymer*length_polymer
frame = gsd.hoomd.Frame()
frame.particles.N = N_particles
frame.particles.typeid = [0] * N_particles
frame.configuration.box = [L, L, L, 0, 0, 0]
frame.particles.types = ['A']


frame.bonds.types = ['bond']

x = np.arange(-L/2.+3, L/2.-3, 1)
y = np.arange(-L/2.+2, L/2.-2, 1)
# Create the 2D meshgrid
X, Y = np.meshgrid(x, y)
coordinates_sq = np.vstack((X.flatten(), Y.flatten())).T
all_bonds = []
all_pos = []

q = 0
for n in range(N_polymer):
    position_z = np.arange(0,length_polymer,1)
    position = np.vstack((np.zeros(length_polymer)+coordinates_sq[q][0],
                          np.zeros(length_polymer)+coordinates_sq[q][1]+5,
                          position_z-3)).T

    q = q+1
    for i,p in enumerate(position):
         if i < length_polymer-1:
            all_bonds.append([i+n*length_polymer,i+1+n*length_polymer])
         all_pos.append(p)

frame.particles.position=all_pos
frame.bonds.N = len(all_bonds)
frame.bonds.group = all_bonds


cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=1)
simulation.create_state_from_snapshot(frame)

integrator = hoomd.md.Integrator(dt=0.005)
all_particles = hoomd.filter.All()
langevin = hoomd.md.methods.Langevin(filter=all_particles, kT=0.5)
simulation.operations.integrator = integrator
integrator.methods.append(langevin)

lj = hoomd.md.pair.LJ(nlist=neigh, default_r_cut=3.0)
lj.mode = "shift"
lj.params[("A","A")] = dict(epsilon=1, sigma=1)
lj.r_cut[("A", "A")] = 2**(1/6.)

integrator.forces =[lj,harmonic]

traj = Trajectory(sequence="A" * length_polymer, num_chains=N_polymer)
traj.frames.append(grab_frame(simulation.state.get_snapshot()))
traj.times.append(simulation.timestep)
n_frames = 24
chunk = max(1, 50000 // n_frames)
for _ in range(n_frames):
    simulation.run(chunk)
    traj.frames.append(grab_frame(simulation.state.get_snapshot()))
    traj.times.append(simulation.timestep)
animate3d(traj)

## Defining Sequence Architectures

TODO: Demo — diblock vs. alternating copolymers

In [ ]:
del simulation

length_polymer = 10
N_polymer = 30
L = 22
sequence = [0,0,0,0,0,0,1,1,1,1]
N_particles = N_polymer*length_polymer
frame = gsd.hoomd.Frame()
frame.particles.N = N_particles
frame.particles.typeid = [0] * N_particles
frame.configuration.box = [L, L, L, 0, 0, 0]
frame.particles.types = ['A','B']


frame.bonds.types = ['bond']

x = np.arange(-L/2.+3, L/2.-3, 1)
y = np.arange(-L/2.+2, L/2.-2, 1)
# Create the 2D meshgrid
X, Y = np.meshgrid(x, y)
coordinates_sq = np.vstack((X.flatten(), Y.flatten())).T
all_bonds = []
all_pos = []
all_types = []
q = 0
for n in range(N_polymer):
    all_types.append(sequence)
    position_z = np.arange(0,length_polymer,1)
    position = np.vstack((np.zeros(length_polymer)+coordinates_sq[q][0],
                          np.zeros(length_polymer)+coordinates_sq[q][1]+5,
                          position_z-3)).T

    q = q+1
    for i,p in enumerate(position):
         if i < length_polymer-1:
            all_bonds.append([i+n*length_polymer,i+1+n*length_polymer])
         all_pos.append(p)


frame.particles.position=all_pos
frame.bonds.N = len(all_bonds)
frame.bonds.group = all_bonds

frame.particles.typeid = np.array(all_types).flatten()


cpu = hoomd.device.CPU()
simulation = hoomd.Simulation(device=cpu, seed=1)
simulation.create_state_from_snapshot(frame)

integrator = hoomd.md.Integrator(dt=0.005)
all_particles = hoomd.filter.All()
langevin = hoomd.md.methods.Langevin(filter=all_particles, kT=0.5)
simulation.operations.integrator = integrator
integrator.methods.append(langevin)

lj = hoomd.md.pair.LJ(nlist=neigh, default_r_cut=3.0)
lj.mode = "shift"
lj.params[("A","A"),("B","B"),("A","B")] = dict(epsilon=1, sigma=1)
lj.r_cut[("A", "A")] = 2**(1/6.)
lj.r_cut[("A", "B")] = 2**(1/6.)
lj.r_cut[("B", "B")] = 3.0

integrator.forces =[lj,harmonic]

gsd_writer = hoomd.write.GSD(
    mode = 'wb',
    trigger=hoomd.trigger.Periodic(100),
    dynamic=['property','momentum'],
    filename='block_copolymer.gsd',
)
simulation.operations.writers.append(gsd_writer)

# themrodyamic quantities
thermodynamic_properties = hoomd.md.compute.ThermodynamicQuantities(
    filter=hoomd.filter.All()
)
simulation.operations.computes.append(thermodynamic_properties)
logger = hoomd.logging.Logger(categories=["scalar"])
logger.add( simulation, quantities=["timestep"])
logger.add(thermodynamic_properties,quantities=["pressure","kinetic_temperature","potential_energy"])
file = open("block_copolymer.log", mode="w")
table = hoomd.write.Table(trigger=hoomd.trigger.Periodic(period=10), logger=logger,output=file)
simulation.operations.writers.append(table)

traj = Trajectory(sequence=sequence, num_chains=N_polymer)
traj.frames.append(grab_frame(simulation.state.get_snapshot()))
traj.times.append(simulation.timestep)
n_frames = 24
chunk = max(1, 50000 // n_frames)
for _ in range(n_frames):
    simulation.run(chunk)
    traj.frames.append(grab_frame(simulation.state.get_snapshot()))
    traj.times.append(simulation.timestep)
animate3d(traj)

Switch out sequence

In [ ]:
sequence = [0,1,0,1,0,1,0,1,0,1]
all_types = []
for n in range(N_polymer):
    all_types.append(sequence)

snapshot = simulation.state.get_snapshot()
snapshot.particles.typeid[:] = np.array(all_types).flatten()
simulation.state.set_snapshot(snapshot)

gsd_writer = hoomd.write.GSD(
    mode = 'wb',
    trigger=hoomd.trigger.Periodic(100),
    dynamic=['property','momentum'],
    filename='alternating_polymer.gsd',
)
simulation.operations.writers.append(gsd_writer)

n_frames = 24
chunk = max(1, 100000 // n_frames)
traj = Trajectory(sequence=sequence, num_chains=N_polymer)
traj.frames.append(grab_frame(simulation.state.get_snapshot()))
traj.times.append(simulation.timestep)
for _ in range(n_frames):
    simulation.run(chunk)
    traj.frames.append(grab_frame(simulation.state.get_snapshot()))
    traj.times.append(simulation.timestep)
animate3d(traj)

## File and Data Management
TODO: what to include here
* Metadata


## References and Further Reading
* Hoomd documentation
